In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import StructType,StructField, StringType,IntegerType,FloatType
catalog_name = 'ecommerce'

## Brands

In [0]:
df_bronze =spark.table(f"{catalog_name}.bronze.brz_brands")
df_bronze.show(10)

In [0]:
df_silver = df_bronze.withColumn('brand_name',F.trim(F.col("brand_name")))
df_silver.show(10)

In [0]:
df_silver = df_silver.withColumn('brand_code',F.regexp_replace('brand_code',r'[^A-Za-z1-9]',''))
df_silver.show(10)

In [0]:
df_silver.select('catagory_code').distinct().show()

In [0]:
# anomalies dictionary
anomalies = {
    "GROCERY": "GRCY",
    "BOOKS": "BKS",
    "TOYS":"TOY"
}
df_silver = df_silver.replace(anomalies,subset='catagory_code')
df_silver.select("catagory_code").distinct().show()

In [0]:
# Write raw data to silver layer(catalog:ecommerce,schema:silver,table:slv_brands)
df_silver.write.format("delta")\
    .mode("overwrite")\
    .option("inferSchema","true")\
    .saveAsTable(f"{catalog_name}.silver.slv_brands")    

## Category

In [0]:
df_bronze = spark.table(f"{catalog_name}.bronze.brz_category")
df_bronze.show(10)

In [0]:
df_duplicates = df_bronze.groupBy('category_code').count().filter(F.col('count')>1)
display(df_duplicates)

In [0]:
df_silver = df_bronze.dropDuplicates(['category_code'])
display(df_silver)

In [0]:
df_silver = df_silver.withColumn('category_code',F.upper(F.col('category_code')))
display(df_silver)

In [0]:
# Write raw data to silver layer(catalog: ecommerce,schema:silver,table:slv_category)
df_silver.write.format("delta")\
    .mode("overwrite")\
    .option("inferSchema","true")\
    .saveAsTable(F"{catalog_name}.silver.slv_category")        

## Products

In [0]:
#Read raw data from bronze table(ecommerce.bronze.brz_products)
df_bronze = spark.read.table(f"{catalog_name}.bronze.brz_products")
# get row and column count
row_count,column_count = df_bronze.count(),len(df_bronze.columns)
# print the results
print(f"Row COunt:{row_count}")
print(F"Column Count:{column_count}")      

In [0]:
%sql
-- Column count must come from catalog metadata (COUNT(*) only counts rows, never columns)
WITH column_meta AS (
  SELECT COUNT(*) AS column_count
  FROM `ecommerce`.information_schema.columns
  WHERE table_schema = 'bronze'
    AND table_name  = 'brz_products'
),
row_meta AS (
  SELECT COUNT(*) AS row_count
  FROM `ecommerce`.`bronze`.`brz_products`
)
SELECT row_count, column_count
FROM row_meta
CROSS JOIN column_meta

In [0]:
%sql

SELECT * FROM ecommerce.silver.slv_brands LIMIT 10

In [0]:
display(df_bronze.limit(10))

In [0]:
df_bronze.select('weight_grams').show(5,truncate=False)

In [0]:
#replace g with ''
df_silver=df_bronze.withColumn('weight_grams',F.regexp_replace(F.col("weight_grams"),"g","").cast(IntegerType()))
df_silver.select('weight_grams').show(5,truncate=False)

####check length_cm(comma instead of dot)

In [0]:
df_silver.select("length_cm").show(3)

In [0]:
#replace, with .
df_silver = df_silver.withColumn('length_cm',F.regexp_replace(F.col("length_cm"),",",".").cast(FloatType()))
df_silver.select('length_cm').show(5,truncate=False)

In [0]:
#category_code and brand_code are in lower case. we need to make it all in uppercase
df_silver.select('category_code','brand_code').show(2)
df_silver = df_silver.withColumn('category_code',F.upper(F.col('category_code')))\
    .withColumn('brand_code',F.upper(F.col('brand_code')))
df_silver.select('category_code','brand_code').show(3)    

In [0]:
# Spelling mistakes in material column
df_silver.select('material').distinct().show()

In [0]:
# Fix spelling Mistakes
df_silver = df_silver.withColumn('material',
F.when(F.col('material')=='Coton','Cotton')
.when(F.col('material')=='Alumium','Aluminum')
.when(F.col('material')=='Ruber','Rubber')
.otherwise(F.col('material'))
)
df_silver.select('material').distinct().show()

In [0]:
#Nagative values in rating count
df_silver.filter(F.col('rating_count')<0).select('rating_count').show(3)

In [0]:
#convert negative rating count to positive
df_silver = df_silver.withColumn('rating_count',F.when(F.col('rating_count').isNotNull(),F.abs(F.col('rating_count')))
.otherwise(F.lit(0)))  
display(df_silver)                             

In [0]:
#check final cleaned data
df_silver.select('weight_grams','length_cm','width_cm','height_cm','rating_count').show(10,truncate=False)

In [0]:
# write raw data to silver_layer(ecommerce.silver.slv_dim_products)
df_silver.write.format('delta')\
.mode('overwrite')\
.option('inferSchema','true') \
.saveAsTable(f"{catalog_name}.silver.slv_dim_products")    

### Customers

In [0]:
#Read the raw data from bronze table(ecommerce.bronze.brz_customers)
df_bronze = spark.read.table(f"{catalog_name}.bronze.brz_customers")
#get row and column count
row_count,column_count = df_bronze.count(),len(df_bronze.columns)
# print the rresults
print(f"Row count:{row_count}")
print(f"column count: {column_count}")
 
df_bronze.show(10)

In [0]:
#handling null values in customer_id column
null_count = df_bronze.filter(F.col('customer_id').isNull()).count()
null_count


In [0]:
df_bronze.filter(F.col('customer_id').isNull()).show(10)

In [0]:
# drop rows where customer_id is null
df_silver = df_bronze.dropna(subset=['customer_id'])
row_count = df_silver.count()
print(f"Row count after dropping null values: {row_count}")

In [0]:
# handle null values in phone columns
null_count = df_silver.filter(F.col('phone').isNull()).count()
null_count

In [0]:
df_silver.filter(F.col('phone').isNull()).show(5)

In [0]:
# fill null values with 'Not Available'
df_silver = df_silver.fillna('Not Available',subset = ['phone'])


In [0]:
#sanity check if any null exists
df_silver.filter(F.col('phone').isNull()).show()

In [0]:
#write raw_data to silver layer(catalog:ecommerce, schema:silver,table:slv_customers)
df_silver.write.format("delta")\
.mode('overwrite')\
.option('inferSchema','true')\
.saveAsTable(f"{catalog_name}.silver.slv_customers")        

### Calendar/Date

In [0]:
# Read the raw data from bronze table(catalog: ecommerce,schema:bronze,table:brz_calendar)
df_bronze = spark.read.table(f"{catalog_name}.bronze.brz_calender")

#Get row and column count
row_count,column_count = df_bronze.count(),len(df_bronze.columns)
print(f" Row count:{row_count}")
print(f" column count:{column_count}")
df_bronze.show(3)

In [0]:
df_bronze.printSchema()

In [0]:
# converting string to date
from pyspark.sql.functions import to_date
df_silver = df_bronze.withColumn('date',to_date(df_bronze["date"],"dd-MM-yyyy"))

In [0]:
print(df_silver.printSchema())
df_silver.show()

In [0]:
# Find duplicates rows in the DataFrame
Duplicates = df_silver.groupBy('date').count().filter('count > 1')

# Show_duplicate Rows
print(f"Total Duplicate rows: {Duplicates.count()}")
display(Duplicates)

In [0]:
#remove duplicate rows
df_silver =df_silver.dropDuplicates(['date'])
# Get row count
row_count = df_silver.count()
print(f" Rows after removing duplicates:" , row_count)

In [0]:
# Day_name normalise casing
df_silver = df_silver.withColumn("day_name",F.initcap(F.col("day_name")))
df_silver.show(5)

In [0]:
# Conver negative week_of the year to positive
df_silver = df_silver.withColumn("week_of_year",F.abs(F.col("week_of_year"))) 
df_silver.show(5)

In [0]:
# Rename a columns
df_silver = df_silver.withColumnRenamed('week_of_year','week')
df_silver.show(3)

In [0]:
# Enhance quarter and  week_of_year column
df_silver = df_silver.withColumn('quarter',F.concat_ws("",F.concat(F.lit('Q'),F.col('quarter'),F.lit('-'),F.col('year'))))
df_silver = df_silver.withColumn('week',F.concat_ws("-",F.concat(F.lit('Week'),F.col('week'),F.lit('-'),F.col('year'))))
df_silver.show(5)


In [0]:
#write raw data to the silver layer(catalog:ecommerce,schema:silver,table:slv_calender)
df_silver.write.format('delta')\
    .mode('overwrite')\
    .option('InferSchema','true')\
    .saveAsTable(f"{catalog_name}.silver.slv_calender")       